# Plotting the results of a simulation for SKA Mid and Low in the AA4 configuration

The following results were obtained using the luminosity prescription from Pardo et al. (2025) and corresponding best-fit parameters based on a spectral index distribution with $\mu=-1.45$ and $\sigma=0.15$. The following plots are based on a single realisation for the Survey Option 3 in Keane et al. (2025) based on the newest SKA survey parameter files.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.colors as colors

rc("text", usetex=True)
rc("font", family="serif")
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 30
MEDIUM_SIZE = 40
BIGGER_SIZE = 60

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [ ]:
colours = ["orange", "#FF5F15", "#0047AB", "purple", "black"]

## Loading the observed pulsar population

In [ ]:
# Read the full ATNF catalog.csv file. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    "../../data/observations/atnf_full_nobinary_24-09-2024_with_errors.csv",
    delimiter=";",
    header=[0, 1],
)
df_atnf.head()

In [ ]:
# Select only those with measured period values.
df_atnf = df_atnf[~df_atnf["P0"]["(s)"].isin(["NAN"])]

# Remove those objects that are in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]
df_atnf = df_atnf[
    ~df_atnf[("ASSOC", "Unnamed: 55_level_1")].str.match("|".join(discard))
]

len(df_atnf)

In [ ]:
df_atnf.head()

In [ ]:
df_atnf.columns = df_atnf.columns.droplevel(1)
df_atnf.head()

In [ ]:
df_atnf = df_atnf.drop(
    columns=[
        "#",
        "PX",
        "POSEPOCH",
        "DM",
        "TAU_SC",
        "S400",
        "S2000",
        "DIST",
        "XX",
        "YY",
    ],
)

len(df_atnf)

In [ ]:
# Select only isolated non-recycled neutron stars through filters with P > 0.01 and Pdot > 1e-19 (for those with measured values).
df_atnf = df_atnf[df_atnf["P0"].to_numpy().astype(np.float64) > 0.01]

len(df_atnf)

Note: For the purpose of modelling the observed isolated population of radio pulsars, in the following, we count those objects with period derivatives larger than $\dot{P} > 10^{-19} s/s$ or those with no measured period derivatives. The latter are likely isolated in nature due to the fact that only a small fraction of ATNF pulsars with known $\dot{P}$ actually have been recycled and attain $\dot{P} < 10^{-19} s/s$. As a result, the following number counts are slighlty larger than the populations used to produce our period period-derivative maps for the simulation-based inference approach.

In [ ]:
df_atnf = df_atnf[
    (df_atnf["P1"].to_numpy().astype(np.float64) > 1.0e-19)
    | (df_atnf["P1"].isin(["NAN"]))
]

In [ ]:
# Extract periods, period derivatives, longitude and latitude for all remaining pulsars.
P_obs = df_atnf["P0"].to_numpy().astype(np.float64)
Pdot_obs = df_atnf["P1"].to_numpy().astype(np.float64)
l_obs = df_atnf["Gl"].to_numpy().astype(np.float64)
b_obs = df_atnf["Gb"].to_numpy().astype(np.float64)

In [ ]:
l_obs[(l_obs > 180.0) & (l_obs < 360.0)] = (
    l_obs[(l_obs > 180.0) & (l_obs < 360.0)] - 360.0
)

In [ ]:
print(len(P_obs))

## Loading the simulated pulsar populations

Load the survey detections plus the corresponding underlying simulated population file for a birth rate of 2.

In [ ]:
df_full = pd.read_pickle(
    f"../../SKA_evol_simulations_si145/all_surveys_newL_br2_2e9/final_population.pkl.gz",
    compression="gzip",
)

In [ ]:
path = "../../SKA_census_simulations_si145/survey_option3/"

df_SKA_low = pd.read_pickle(
    f"{path}/run_1/survey_SKA_low_AA4_results.pkl.gz",
    compression="gzip",
)

df_SKA_mid2 = pd.read_pickle(
    f"{path}/run_1/survey_SKA_mid_band2_AA4_results.pkl.gz",
    compression="gzip",
)

df_SKA_low.head()

Export df with partial info.

In [ ]:
# df_SKA_low = df_SKA_low.copy()
# df_SKA_mid2 = df_SKA_mid.copy()

# df_SKA_low = df_SKA_low.drop(
#     columns=[
#         "RA",
#         "DEC",
#         "pm_RA",
#         "pm_DEC",
#         "P",
#         "P_dot",
#         "w_eff",
#     ],
#     axis=1,
#     level=0,
# )
# df_SKA_mid2 = df_SKA_mid.drop(
#     columns=[
#         "RA",
#         "DEC",
#         "pm_RA",
#         "pm_DEC",
#         "P",
#         "P_dot",
#         "w_eff",
#     ],
#     axis=1,
#     level=0,
# )

# df_SKA_low.to_csv("SKA_low_AA4.csv", index=False)
# df_SKA_mid2.to_csv("SKA_mid2_AA4.csv", index=False)

In [ ]:
x = df_full["x"]["[kpc]"].to_numpy()
y = df_full["y"]["[kpc]"].to_numpy()
z = df_full["z"]["[kpc]"].to_numpy()
RA = df_full["RA"]["[deg]"].to_numpy()
DEC = df_full["DEC"]["[deg]"].to_numpy()
pm_RA = df_full["pm_RA"]["[mas yr^-1]"].to_numpy()
pm_DEC = df_full["pm_DEC"]["[mas yr^-1]"].to_numpy()
l = df_full["l"]["[deg]"].to_numpy()
b = df_full["b"]["[deg]"].to_numpy()
v_r = df_full["v_r"]["[km s^-1]"].to_numpy()
v_phi = df_full["v_phi"]["[km s^-1]"].to_numpy()
v_z = df_full["v_z"]["[km s^-1]"].to_numpy()
dist = df_full["d"]["[kpc]"].to_numpy()
B = df_full["B"]["[G]"].to_numpy()
chi = df_full["chi"]["[rad]"].to_numpy()
P = df_full["P"]["[s]"].to_numpy()
P_dot = df_full["P_dot"]["[s s^-1]"].to_numpy()
L_radio_bol = df_full["L_radio_bol"]["[erg s^-1]"].to_numpy()
w_int = df_full["w_int"]["[s]"].to_numpy()
intercepted_radio = df_full["intercepted_radio"][" "].to_numpy(dtype=bool)
age = df_full["age"]["[yr]"].to_numpy()

In [ ]:
idx_SKA_low = df_SKA_low["NS_idx"][" "].to_numpy(dtype=int)
idx_SKA_mid2 = df_SKA_mid2["NS_idx"][" "].to_numpy(dtype=int)

set_SKA_low = set(idx_SKA_low.tolist())
set_SKA_mid2 = set(idx_SKA_mid2.tolist())

In [ ]:
P_SKA_low = df_SKA_low["P"]["[s]"].to_numpy()
print(P_SKA_low[:10])
print(P[idx_SKA_low][:10])

## Plotting the positional distributions

Top view of the Galactic plane.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 10))

ax.plot(
    x,
    y,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Simulated all",
)
ax.plot(
    x[intercepted_radio],
    y[intercepted_radio],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=0.1,
    rasterized=True,
    label="Intercepting our LOS",
)
ax.plot(
    x[idx_SKA_low],
    y[idx_SKA_low],
    linestyle="None",
    marker="o",
    color=colours[0],
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SKA-Low",
)
ax.plot(
    x[idx_SKA_mid2],
    y[idx_SKA_mid2],
    linestyle="None",
    marker="o",
    color=colours[1],
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SKA-Mid Band 2",
)

ax.plot(0.0, 8.3, marker="o", color="tab:red", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.legend(bbox_to_anchor=(1, 0.7), frameon=False, loc=0, fontsize=20)

plt.show()

Side view of the Galactic plane.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

ax.plot(
    x,
    z,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Simulated all",
)
ax.plot(
    x[intercepted_radio],
    z[intercepted_radio],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=0.1,
    rasterized=True,
    label="Intercepting our LOS",
)
ax.plot(
    x[idx_SKA_low],
    z[idx_SKA_low],
    linestyle="None",
    marker="o",
    color=colours[0],
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SKA-Low",
)
ax.plot(
    x[idx_SKA_mid2],
    z[idx_SKA_mid2],
    linestyle="None",
    marker="o",
    color=colours[1],
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SKA-Mid Band 2",
)

ax.plot(0.0, 0.02, marker="o", color="tab:red", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.legend(bbox_to_anchor=(1, 0.7), frameon=False, loc=0, fontsize=20)

plt.show()

## Plotting the distribution along z

In [ ]:
bins_z_full_range = np.linspace(-20, 20, 151)

z_values = np.linspace(0, 20, 501)
scale_height_psrpoppy = 0.33
pdf_z_psrpoppy = (
    1.0 / scale_height_psrpoppy * np.exp(-z_values / scale_height_psrpoppy)
)

In [ ]:
z_observed = df_atnf["ZZ"].to_numpy()
z_SKA_low = z[idx_SKA_low]
z_SKA_mid2 = z[idx_SKA_mid2]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    z_observed,
    bins=bins_z_full_range,
    histtype="step",
    color=colours[2],
    lw=4,
    alpha=0.8,
    label=r"Observed full",
    density=True,
)
ax.hist(
    z_SKA_low,
    bins=bins_z_full_range,
    histtype="step",
    color=colours[0],
    lw=4,
    alpha=0.8,
    label=r"SKA-Low",
    density=True,
)
ax.hist(
    z_SKA_mid2,
    bins=bins_z_full_range,
    histtype="step",
    color=colours[1],
    lw=4,
    alpha=0.8,
    label=r"SKA-Mid Band 2",
    density=True,
)

ax.plot(
    z_values,
    pdf_z_psrpoppy,
    color="black",
    lw=4,
    alpha=0.8,
    label=r"Exp. PDF psrpoppy $h_c = 0.33\,$kpc",
)
ax.plot(
    -z_values,
    pdf_z_psrpoppy,
    color="black",
    lw=4,
    alpha=0.8,
)

plt.xlabel(r"z [kpc]")
plt.ylabel(r"PDF of detected stars")
ax.set_xlim(-5.0, 5.0)
ax.set_ylim(0.0, 2.0)
plt.legend(frameon=True, loc=1)

plt.show()

### Fitting an exponential PDF to SKA detections

Make histogram for all the SKA detections and fit an exponential PDF. For Survey Option 3, there is no overlap between the two bands.

In [ ]:
combined_idx = np.concatenate([idx_SKA_low, idx_SKA_mid2])

z_combined = z[combined_idx]
print(len(z_combined))

In [ ]:
z_combined

Fit the exponential to the positive and negative z-values separately as the zenith dependence in Low means the detected distributions are not unique across the Galactic plane.

In [ ]:
lambda_pos = np.mean(z_combined[z_combined > 0])
lambda_neg = np.abs(np.mean(z_combined[z_combined < 0]))

print(lambda_pos)
print(lambda_neg)

In [ ]:
scale_height_pos = np.round(lambda_pos, 2)
scale_height_neg = np.round(lambda_neg, 2)

z_values = np.linspace(0, 20, 501)
pdf_z_pos = 1.0 / scale_height_pos * np.exp(-z_values / scale_height_pos)
pdf_z_neg = 1.0 / scale_height_neg * np.exp(-z_values / scale_height_neg)

In [ ]:
bins_z = np.linspace(0, 15, 51)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    z_combined[z_combined > 0],
    bins=bins_z,
    histtype="step",
    color=colours[3],
    lw=4,
    alpha=0.8,
    label=r"SKA combined Option 3",
    density=True,
)
ax.hist(
    z_combined[z_combined < 0],
    # Flip the entries at 0 to make sure they increase monotonically.
    bins=-bins_z[::-1],
    histtype="step",
    color=colours[3],
    lw=4,
    alpha=0.8,
    density=True,
)

ax.plot(
    z_values,
    pdf_z_pos,
    color="pink",
    lw=4,
    alpha=0.8,
    label=f"Exp. PDF pos $h_c = {scale_height_pos}\,$kpc",
)
ax.plot(
    -z_values,
    pdf_z_neg,
    color="pink",
    lw=4,
    alpha=0.8,
    label=f"Exp. PDF neg $h_c = {scale_height_neg}\,$kpc",
)

plt.xlabel(r"z [kpc]")
plt.ylabel(r"PDF of detected stars")
ax.set_xlim(-10.0, 10.0)
ax.set_ylim(0.0, 0.9)
plt.legend(frameon=True, loc=1)

plt.show()

### Fitting a stretched exponential to the full simulation data

Histogram for all simulated stars with fitted stretched exponential.

In [ ]:
len(z)

In [ ]:
# Split into positive and negative parts.
z_pos = z[z > 0]
z_neg = -z[z < 0]  # Flip negatives to fit as positives.

z_pos = z[z > 0]
z_neg = -z[z < 0]  # Flip negatives to fit as positives.

In [ ]:
# Define histogram values and bin extents.
bins_z_full = np.linspace(0.0, 25, 151)
counts_pos, bin_edges_pos = np.histogram(z_pos, bins=bins_z_full, density=True)
bin_centers_pos = (bin_edges_pos[:-1] + bin_edges_pos[1:]) / 2

counts_neg, bin_edges_neg = np.histogram(z_neg, bins=bins_z_full, density=True)
bin_centers_neg = (bin_edges_neg[:-1] + bin_edges_neg[1:]) / 2

In [ ]:
from scipy.optimize import curve_fit

In [ ]:
# PDF for the Weibull distribution.
def double_exponential(x, w, lambda_1, lambda_2):
    term_1 = w / lambda_1 * np.exp(-x / lambda_1)
    term_2 = (1 - w) / lambda_2 * np.exp(-x / lambda_2)
    return term_1 + term_2

In [ ]:
# Fit the values with an initial guess.
w_pos, lambda_1_pos, lambda_2_pos = curve_fit(
    double_exponential, bin_centers_pos, counts_pos
)[0]
print(w_pos, lambda_1_pos, lambda_2_pos)

w_neg, lambda_1_neg, lambda_2_neg = curve_fit(
    double_exponential, bin_centers_neg, counts_neg
)[0]
print(w_neg, lambda_1_neg, lambda_2_neg)

In [ ]:
# As values are identical for above and below the Galactic plane, we focus on the positive ones only.
w_pos_rounded = np.round(w_pos, 2)
lambda_1_pos_rounded = np.round(lambda_1_pos, 2)
lambda_2_pos_rounded = np.round(lambda_2_pos, 2)

In [ ]:
# Determine PDF for plotting.
z_values_full = np.linspace(0.0, 250, 2001)
pdf_double_exp = double_exponential(
    z_values_full, w_pos, lambda_1_pos, lambda_2_pos
)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

ax.hist(
    z_pos,
    bins=bins_z_full,
    histtype="step",
    color="grey",
    lw=4,
    alpha=0.8,
    label=r"Isolated simulated population: $4\times 10^7$ NSs",
    density=True,
)
ax.plot(
    z_values_full,
    pdf_double_exp,
    color="black",
    lw=4,
    alpha=0.8,
    label=f"Double exp. $P(z) = {w_pos / lambda_1_pos:.2f} e^{{-z/{lambda_1_pos:.2f}}} + {(1-w_pos) / lambda_2_pos:.2f} e^{{-z/{lambda_2_pos:.2f}}}$",
)

plt.xlabel(r"z [kpc]")
plt.ylabel(r"PDF of detected stars")
ax.set_xlim(0.0, 15.0)
ax.set_ylim(0.0, 0.6)
plt.legend(frameon=True, loc=1)

plt.show()

### Sampling from stretched exponential to check validity

In [ ]:
from scipy import integrate
from scipy import interpolate

# Calculating CDF with trapezoidal rule.
CDF = integrate.cumtrapz(pdf_double_exp, z_values_full, initial=0)
CDF_norm = CDF / max(CDF)
print(CDF_norm[-1])

In [ ]:
# Sampling from CDF to get certain number of random samples.
n_samples = 40000
CDF_random = np.random.uniform(0.0, 1, n_samples)
CDF_interp = interpolate.interp1d(CDF, z_values_full)
z_full_sampled = CDF_interp(CDF_random)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

ax.hist(
    z_pos,
    bins=bins_z_full,
    histtype="step",
    color="grey",
    lw=4,
    alpha=0.8,
    label=r"Isolated simulated population",
    density=True,
)

ax.hist(
    z_full_sampled,
    bins=bins_z_full,
    histtype="step",
    color="blue",
    lw=4,
    alpha=0.8,
    label=r"Sampled from fitted PDF",
    density=True,
)

plt.xlabel(r"z [kpc]")
plt.ylabel(r"PDF of detected stars")
ax.set_xlim(0.0, 10.0)
ax.set_ylim(0.0, 0.6)
plt.legend(frameon=True, loc=1)

plt.show()

Comparison of two distributions with KS test.

In [ ]:
import scipy.stats as st

st.kstest(z_pos, z_full_sampled)

In [ ]:
sample_lognormal_A = np.random.lognormal(mean=10.0, sigma=2.0, size=50)
sample_lognormal_B = np.random.lognormal(mean=10.0, sigma=2.0, size=40)

st.kstest(sample_lognormal_A, sample_lognormal_B)

KS test shows that they are not from the same distribution, but this is overall the best we can do with an analytical prescription for the PDF.

## Plotting the positional distribution

In [ ]:
l_SKA_low = df_SKA_low["l"]["[deg]"].to_numpy()
b_SKA_low = df_SKA_low["b"]["[deg]"].to_numpy()
l_SKA_mid2 = df_SKA_mid2["l"]["[deg]"].to_numpy()
b_SKA_mid2 = df_SKA_mid2["b"]["[deg]"].to_numpy()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))

plt.plot(
    l_SKA_low,
    b_SKA_low,
    linestyle="None",
    marker="o",
    color=colours[0],
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Simulated SKA-Low",
)

plt.plot(
    l_SKA_mid2,
    b_SKA_mid2,
    linestyle="None",
    marker="o",
    color=colours[1],
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Simulated SKA-Mid Band 2",
)

plt.plot(
    l_obs,
    b_obs,
    linestyle="None",
    marker="o",
    color=colours[2],
    markersize=5,
    alpha=1,
    rasterized=True,
    label=r"Observed population",
)

ax.set_xlim(-190.0, 190.0)
ax.set_ylim(-95.0, 95.0)
ax.set_xlabel(r"Galactic longitude $l$ [deg]")
ax.set_ylabel(r"Galactic latitude $b$ [deg]")
ax.legend(frameon=True, loc="best")
ax.grid()

plt.tight_layout()
plt.show()

In [ ]:
# Convert Galactic longitude to radians for the Aitoff projection.
l_obs_rad = np.radians(l_obs)
l_SKA_low_rad = np.radians(l_SKA_low)
l_SKA_mid2_rad = np.radians(l_SKA_mid2)

# Convert Galactic latitude to radians.
b_obs_rad = np.radians(b_obs)
b_SKA_low_rad = np.radians(b_SKA_low)
b_SKA_mid2_rad = np.radians(b_SKA_mid2)

fig, ax = plt.subplots(figsize=(15, 10), subplot_kw={"projection": "aitoff"})
ax.set_title(f"AA4 configuration", fontsize=MEDIUM_SIZE)

ax.scatter(
    l_SKA_low_rad,
    b_SKA_low_rad,
    marker="o",
    color=colours[0],
    s=40,
    alpha=1,
    label=r"Simulated SKA-Low",
)

ax.scatter(
    l_SKA_mid2_rad,
    b_SKA_mid2_rad,
    marker="o",
    color=colours[1],
    s=40,
    alpha=1,
    label=r"Simulated SKA-Mid Band 2",
)

ax.scatter(
    l_obs_rad,
    b_obs_rad,
    marker="o",
    color=colours[2],
    s=40,
    alpha=1,
    label=r"Observed population",
)

ax.set_xticks(np.radians(np.linspace(-180, 180, 13)))
ax.set_xticklabels(
    [
        "",
        "",
        r"$-120^\circ$",
        "",
        r"$-60^\circ$",
        "",
        r"$0^\circ$",
        "",
        r"$60^\circ$",
        "",
        r"$120^\circ$",
        "",
        "",
    ]
)
ax.set_yticks(np.radians(np.linspace(-90, 60, 6)))
ax.set_yticklabels(
    [
        r"$-90^\circ$",
        r"$-60^\circ$",
        r"$-30^\circ$",
        r"$0^\circ$",
        r"$30^\circ$",
        r"$60^\circ$",
        # r"$90^\circ$",
    ]
)

ax.grid(True)  # Add grid lines
ax.legend(frameon=True, bbox_to_anchor=(0.7, 0.95))

plt.tight_layout()
plt.show()

## Plotting the DM distribution

In [ ]:
DM_SKA_mid2 = df_SKA_mid2["DM"]["[pc cm^-3]"].to_numpy()
DM_SKA_low = df_SKA_low["DM"]["[pc cm^-3]"].to_numpy()
P_SKA_low = df_SKA_low["P"]["[s]"].to_numpy()
P_SKA_mid2 = df_SKA_mid2["P"]["[s]"].to_numpy()
P_dot_SKA_low = df_SKA_low["P_dot"]["[s s^-1]"].to_numpy()
P_dot_SKA_mid2 = df_SKA_mid2["P_dot"]["[s s^-1]"].to_numpy()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
ax.set_title(f"AA4 configuration", fontsize=MEDIUM_SIZE)

ax.scatter(
    P_SKA_mid2,
    DM_SKA_mid2,
    marker="o",
    color=colours[0],
    s=40,
    alpha=1,
    label=r"Simulated SKA-Mid Band 2",
)

ax.scatter(
    P_SKA_low,
    DM_SKA_low,
    marker="o",
    color=colours[1],
    s=40,
    alpha=1,
    label=r"Simulated SKA-Low",
)

ax.set_xscale("log")

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"DM [pc/cm$^3$]")
plt.legend(frameon=True, loc=1)

plt.show()

## Plotting a PPdot diagram and corresponding histograms

Edot lines.

In [ ]:
I_NS = 1.36e45
R_NS = 1.1e6  # in cm.
M_NS = 1.4  # in solar masses.
c = 2.998e10

Pdot_Edot_lines = np.zeros((6, 51))

Edot_log = np.linspace(28, 38, 6)
print(Edot_log)

In [ ]:
P_log = np.linspace(-3, 2, 51)

In [ ]:
for i in range(len(Edot_log)):
    Pdot_Edot_lines[i] = (
        10 ** Edot_log[i] * (10**P_log) ** 3 / (4 * np.pi**2 * I_NS)
    )

B lines.

In [ ]:
Pdot_B_lines = np.zeros((5, 51))

B_log = np.linspace(10, 14, 5)
print(B_log)

In [ ]:
for i in range(len(B_log)):
    Pdot_B_lines[i] = (
        np.pi**2
        * (10 ** B_log[i]) ** 2
        * (R_NS**6)
        / (I_NS * 10**P_log * c**3)
    )

Deathlines.

In [ ]:
P_dot_DL_1 = (
    1.2e-15
    * (1.4 / 1) ** (-1)
    * (R_NS / 1e6) ** (-3 / 4)
    * (10**P_log) ** (11 / 4)
)
b = 10
P_dot_DL_2 = 2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (10**P_log) ** 2

In [ ]:
fig, ax = plt.subplots(figsize=(12, 14))
ax.set_title(f"AA4 configuration", fontsize=MEDIUM_SIZE)

for i in range(len(Edot_log)):
    ax.plot(
        10**P_log,
        Pdot_Edot_lines[i],
        linestyle="-",
        color="gray",
        alpha=0.5,
        rasterized=True,
    )
for i in range(len(B_log)):
    ax.plot(
        10**P_log,
        Pdot_B_lines[i],
        linestyle="-",
        color="gray",
        alpha=0.5,
        rasterized=True,
    )

ax.plot(
    10**P_log,
    P_dot_DL_1,
    linestyle="-",
    linewidth=3.0,
    color="black",
    alpha=0.7,
    rasterized=True,
)
ax.plot(
    10**P_log,
    P_dot_DL_2,
    linestyle="-",
    linewidth=3.0,
    color="black",
    alpha=0.7,
    rasterized=True,
)
ax.fill_between(10**P_log, P_dot_DL_1, P_dot_DL_2, color="grey", alpha=0.3)

ax.scatter(
    P[idx_SKA_low],
    P_dot[idx_SKA_low],
    marker="o",
    color=colours[0],
    s=40,
    alpha=1,
    label=r"SKA-Low",
)

ax.scatter(
    P[idx_SKA_mid2],
    P_dot[idx_SKA_mid2],
    marker="o",
    color=colours[1],
    s=40,
    alpha=1,
    label=r"SKA-Mid Band 2",
)

ax.scatter(
    P_obs,
    Pdot_obs,
    marker="o",
    color=colours[2],
    s=40,
    alpha=1,
    label=r"Observed",
)

ax.text(
    0.412,
    0.015,
    r"$10^{28} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=53,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.275,
    0.015,
    r"$10^{30} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=53,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.143,
    0.015,
    r"$10^{32} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=53,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.015,
    r"$10^{34} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=53,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.17,
    r"$10^{36} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=53,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.325,
    r"$10^{38} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=53,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)

ax.text(
    0.9,
    0.027,
    r"$10^{10} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.9,
    0.182,
    r"$10^{11} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.9,
    0.335,
    r"$10^{12} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.9,
    0.49,
    r"$10^{13} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.9,
    0.645,
    r"$10^{14} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(1.2e-3, 100.0)
ax.set_ylim(1.0e-22, 1.0e-9)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
plt.legend(frameon=True, loc=2)

plt.tight_layout()
plt.savefig("SKA_ppdot_AA4.pdf", dpi=300, bbox_inches="tight")
plt.show()

Observed and SKA numbers above lower deathline.

In [ ]:
P_dot_DL_obs_cutoff = 2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (P_obs) ** 2
print(sum(Pdot_obs < P_dot_DL_obs_cutoff))  # Below as a check

P_dot_DL_SKA_low_cutoff = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (P_SKA_low) ** 2
)
print("Low above:", sum(P_dot_SKA_low > P_dot_DL_SKA_low_cutoff))
print("Low below:", sum(P_dot_SKA_low < P_dot_DL_SKA_low_cutoff))

P_dot_DL_SKA_mid2_cutoff = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (P_SKA_mid2) ** 2
)

print("Mid Band 2 above:", sum(P_dot_SKA_mid2 > P_dot_DL_SKA_mid2_cutoff))
print("Mid Band 2 below:", sum(P_dot_SKA_mid2 < P_dot_DL_SKA_mid2_cutoff))

In [ ]:
# Define bins for the histogram x-axes.
bins_period = np.linspace(-2.4, 2.4, 25)
print(bins_period)

bins_period_deriv = np.linspace(-22, -10, 21)
print(bins_period_deriv)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(P_obs),
    bins=bins_period,
    histtype="step",
    color=colours[2],
    lw=4,
    alpha=0.8,
    label=r"Observed population",
)
ax.hist(
    np.log10(P[idx_SKA_low]),
    bins=bins_period,
    histtype="step",
    color=colours[0],
    lw=4,
    alpha=0.8,
    label=r"SKA-Low",
)
ax.hist(
    np.log10(P[idx_SKA_mid2]),
    bins=bins_period,
    histtype="step",
    color=colours[1],
    lw=4,
    alpha=0.8,
    label=r"SKA-Mid Band 2",
)

ax.set_yscale("log")

plt.xlabel(r"log$_{10} P$ [s]")
plt.ylabel(r"PDF of detected stars")
plt.legend(frameon=True, loc=1)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(P_dot[idx_SKA_low]),
    bins=bins_period_deriv,
    histtype="step",
    color=colours[0],
    lw=4,
    alpha=0.8,
    label=r"SKA-Low",
)
ax.hist(
    np.log10(P_dot[idx_SKA_mid2]),
    bins=bins_period_deriv,
    histtype="step",
    color=colours[1],
    lw=4,
    alpha=0.8,
    label=r"SKA-Mid Band 2",
)
ax.hist(
    np.log10(Pdot_obs),
    bins=bins_period_deriv,
    histtype="step",
    color=colours[2],
    lw=4,
    alpha=0.8,
    label=r"Observed population",
)

ax.set_yscale("log")

plt.xlabel(r"log$_{10} \dot{P}$ [s/s]")
plt.ylabel(r"PDF of detected stars")
plt.legend(frameon=True, loc=1)

plt.show()

## Proper motion plots

### Observed population

In [ ]:
# Extract ICRS coordinates, proper motions and distances for the observed sample.
RA_obs = df_atnf["RAJD"].to_numpy().astype(np.float64)
DEC_obs = df_atnf["DECJD"].to_numpy().astype(np.float64)
pm_RA_obs = df_atnf["PMRA"].to_numpy().astype(np.float64)
pm_DEC_obs = df_atnf["PMDEC"].to_numpy().astype(np.float64)
dist_obs = df_atnf["DIST_DM"].to_numpy().astype(np.float64)

In [ ]:
DEG_TO_MAS = 3600000  # Convert [deg] to [mas]
RA_galcen = 266.4  # Galactic center position in ra [deg]
DEC_galcen = -29.0  # Galactic center position in dec [deg]

In [ ]:
RA_vel_obs = RA_obs[~np.isnan(pm_DEC_obs)]
DEC_vel_obs = DEC_obs[~np.isnan(pm_DEC_obs)]
dist_vel_obs = dist_obs[~np.isnan(pm_DEC_obs)]
pm_RA_vel_obs = pm_RA_obs[~np.isnan(pm_DEC_obs)]
pm_DEC_vel_obs = pm_DEC_obs[~np.isnan(pm_DEC_obs)]

Projected trajectories.

In [ ]:
# Plot the trajectories of the observed neutron stars in the equatorial frame over the past 0.5 Myr.
t = np.linspace(0.0, 0.5e6, 100)

RA_obs_traj = np.zeros([len(RA_vel_obs), len(t)])
DEC_obs_traj = np.zeros([len(RA_vel_obs), len(t)])

for i in range(len(RA_vel_obs)):
    RA_obs_traj[i, :] = RA_vel_obs[i] - pm_RA_vel_obs[i] * t / DEG_TO_MAS
    DEC_obs_traj[i, :] = DEC_vel_obs[i] - pm_DEC_vel_obs[i] * t / DEG_TO_MAS

In [ ]:
np.array_equal(RA_obs_traj[:, 0], RA_vel_obs)

In [ ]:
np.array_equal(DEC_obs_traj[:, 0], DEC_vel_obs)

Determine proper motion magnitude to overplot corresponding colour.

In [ ]:
pm_mag_obs = np.sqrt(pm_RA_vel_obs**2 + pm_DEC_vel_obs**2)

print(pm_mag_obs[:10])

In [ ]:
# Custom color map for observations.
color_obs_start = "#82c7ff"
color_obs_end = colours[2]
obs_cmap = mpl.colors.LinearSegmentedColormap.from_list(
    "custom_cmap", [color_obs_start, color_obs_end]
)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))

# Plot data.
ax.scatter(
    RA_obs,
    DEC_obs,
    marker="o",
    facecolors="gray",
    s=20,
    alpha=0.2,
    rasterized=True,
    zorder=0,
)
scatter = ax.scatter(
    RA_vel_obs,
    DEC_vel_obs,
    linestyle="None",
    marker="o",
    c=pm_mag_obs,
    cmap=obs_cmap,
    s=40,
    alpha=1.0,
    rasterized=True,
    vmin=1,
    vmax=390,
)
# Create colorbar axes.
cbar_ax = fig.add_axes([0.915, 0.11, 0.025, 0.77])

# Create the colorbars.
cbar_obs = fig.colorbar(scatter, cax=cbar_ax)

# Set label on one colorbar only.
cbar_obs.set_label(r"$\mu_{\rm tot}$ [mas yr$^{-1}$]")

# Plot trajectories.
for i in range(len(RA_vel_obs)):
    ax.plot(
        RA_obs_traj[i],
        DEC_obs_traj[i],
        linestyle="-",
        linewidth=1.5,
        color="black",
        alpha=1,
        rasterized=True,
    )

# Add Galactic centre.
ax.plot(
    RA_galcen,
    DEC_galcen,
    marker="*",
    markeredgewidth=1,
    color="#800020",
    markersize=25,
    zorder=1,
)

ax.set_xlim(0.0, 360.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("DEC [deg]")

plt.savefig("pm_obs.jpg", dpi=300, bbox_inches="tight")
plt.show()

### SKA simulations

In [ ]:
# Extract values for SKA surveys.
dist_SKA_mid2 = df_SKA_mid2["d"]["[kpc]"].to_numpy().astype(np.float64)
RA_SKA_mid2 = df_SKA_mid2["RA"]["[deg]"].to_numpy().astype(np.float64)
DEC_SKA_mid2 = df_SKA_mid2["DEC"]["[deg]"].to_numpy().astype(np.float64)
pm_RA_SKA_mid2 = (
    df_SKA_mid2["pm_RA"]["[mas yr^-1]"].to_numpy().astype(np.float64)
)
pm_DEC_SKA_mid2 = (
    df_SKA_mid2["pm_DEC"]["[mas yr^-1]"].to_numpy().astype(np.float64)
)

dist_SKA_low = df_SKA_low["d"]["[kpc]"].to_numpy().astype(np.float64)
RA_SKA_low = df_SKA_low["RA"]["[deg]"].to_numpy().astype(np.float64)
DEC_SKA_low = df_SKA_low["DEC"]["[deg]"].to_numpy().astype(np.float64)
pm_RA_SKA_low = (
    df_SKA_low["pm_RA"]["[mas yr^-1]"].to_numpy().astype(np.float64)
)
pm_DEC_SKA_low = (
    df_SKA_low["pm_DEC"]["[mas yr^-1]"].to_numpy().astype(np.float64)
)

In [ ]:
# Only select those above the lower death line.
dist_SKA_mid2 = dist_SKA_mid2[P_dot_SKA_mid2 > P_dot_DL_SKA_mid2_cutoff]
RA_SKA_mid2 = RA_SKA_mid2[P_dot_SKA_mid2 > P_dot_DL_SKA_mid2_cutoff]
DEC_SKA_mid2 = DEC_SKA_mid2[P_dot_SKA_mid2 > P_dot_DL_SKA_mid2_cutoff]
pm_RA_SKA_mid2 = pm_RA_SKA_mid2[P_dot_SKA_mid2 > P_dot_DL_SKA_mid2_cutoff]
pm_DEC_SKA_mid2 = pm_DEC_SKA_mid2[P_dot_SKA_mid2 > P_dot_DL_SKA_mid2_cutoff]

dist_SKA_low = dist_SKA_low[P_dot_SKA_low > P_dot_DL_SKA_low_cutoff]
RA_SKA_low = RA_SKA_low[P_dot_SKA_low > P_dot_DL_SKA_low_cutoff]
DEC_SKA_low = DEC_SKA_low[P_dot_SKA_low > P_dot_DL_SKA_low_cutoff]
pm_RA_SKA_low = pm_RA_SKA_low[P_dot_SKA_low > P_dot_DL_SKA_low_cutoff]
pm_DEC_SKA_low = pm_DEC_SKA_low[P_dot_SKA_low > P_dot_DL_SKA_low_cutoff]

print(len(dist_SKA_low))

SKA trajectories.

In [ ]:
RA_SKA_mid2_traj = np.zeros([len(RA_SKA_mid2), len(t)])
DEC_SKA_mid2_traj = np.zeros([len(RA_SKA_mid2), len(t)])

for i in range(len(RA_SKA_mid2)):
    RA_SKA_mid2_traj[i, :] = (
        RA_SKA_mid2[i] - pm_RA_SKA_mid2[i] * t / DEG_TO_MAS
    )
    DEC_SKA_mid2_traj[i, :] = (
        DEC_SKA_mid2[i] - pm_DEC_SKA_mid2[i] * t / DEG_TO_MAS
    )

In [ ]:
RA_SKA_low_traj = np.zeros([len(RA_SKA_low), len(t)])
DEC_SKA_low_traj = np.zeros([len(RA_SKA_low), len(t)])

for i in range(len(RA_SKA_low)):
    RA_SKA_low_traj[i, :] = RA_SKA_low[i] - pm_RA_SKA_low[i] * t / DEG_TO_MAS
    DEC_SKA_low_traj[i, :] = (
        DEC_SKA_low[i] - pm_DEC_SKA_low[i] * t / DEG_TO_MAS
    )

Proper motion magnitudes.

In [ ]:
pm_mag_SKA_low = np.sqrt(pm_RA_SKA_low**2 + pm_DEC_SKA_low**2)
pm_mag_SKA_mid2 = np.sqrt(pm_RA_SKA_mid2**2 + pm_DEC_SKA_mid2**2)

print(pm_mag_SKA_low[:10])
print(min(pm_mag_SKA_low), max(pm_mag_SKA_low))
print(pm_mag_SKA_mid2[:10])
print(min(pm_mag_SKA_mid2), max(pm_mag_SKA_mid2))

In [ ]:
# Custom color map for observations.
color_SKA_low_start = colours[0]
color_SKA_low_end = "#004953"
SKA_low_cmap = mpl.colors.LinearSegmentedColormap.from_list(
    "custom_cmap", [color_SKA_low_start, color_SKA_low_end]
)

color_SKA_mid2_start = colours[1]
color_SKA_mid2_end = "#301934"
color_SKA_mid2_end = "#301934"
SKA_mid2_cmap = mpl.colors.LinearSegmentedColormap.from_list(
    "custom_cmap", [color_SKA_mid2_start, color_SKA_mid2_end]
)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))

# Plot data.
scatter_mid2 = ax.scatter(
    RA_SKA_mid2[(pm_mag_SKA_mid2 > 5) & (pm_mag_SKA_mid2 < 50)],
    DEC_SKA_mid2[(pm_mag_SKA_mid2 > 5) & (pm_mag_SKA_mid2 < 50)],
    linestyle="None",
    marker="o",
    c=pm_mag_SKA_mid2[(pm_mag_SKA_mid2 > 5) & (pm_mag_SKA_mid2 < 50)],
    cmap=SKA_mid2_cmap,
    s=40,
    alpha=1.0,
    rasterized=True,
    vmin=5,
    vmax=50,
)
ax.scatter(
    RA_SKA_mid2[pm_mag_SKA_mid2 < 5],
    DEC_SKA_mid2[pm_mag_SKA_mid2 < 5],
    linestyle="None",
    marker="o",
    color="gray",
    s=40,
    alpha=0.2,
    rasterized=True,
)
ax.scatter(
    RA_SKA_mid2[pm_mag_SKA_mid2 > 50],
    DEC_SKA_mid2[pm_mag_SKA_mid2 > 50],
    linestyle="None",
    marker="o",
    color="black",
    s=40,
    alpha=1.0,
    rasterized=True,
)
scatter_low = ax.scatter(
    RA_SKA_low[(pm_mag_SKA_low > 5) & (pm_mag_SKA_low < 50)],
    DEC_SKA_low[(pm_mag_SKA_low > 5) & (pm_mag_SKA_low < 50)],
    linestyle="None",
    marker="o",
    c=pm_mag_SKA_low[(pm_mag_SKA_low > 5) & (pm_mag_SKA_low < 50)],
    cmap=SKA_low_cmap,
    s=40,
    alpha=1.0,
    rasterized=True,
    vmin=5,
    vmax=50,
)
ax.scatter(
    RA_SKA_low[pm_mag_SKA_low < 5],
    DEC_SKA_low[pm_mag_SKA_low < 5],
    linestyle="None",
    marker="o",
    color="gray",
    s=40,
    alpha=0.2,
    rasterized=True,
)
ax.scatter(
    RA_SKA_low[pm_mag_SKA_low > 50],
    DEC_SKA_low[pm_mag_SKA_low > 50],
    linestyle="None",
    marker="o",
    color="black",
    s=40,
    alpha=1.0,
    rasterized=True,
)
# Create colorbar axes.
cbar_ax1 = fig.add_axes([0.945, 0.11, 0.025, 0.77])  # First colorbar
cbar_ax2 = fig.add_axes([0.915, 0.11, 0.025, 0.77])  # Second colorbar

# Create the colorbars.
cbar_low = fig.colorbar(scatter_low, cax=cbar_ax1)
cbar_mid2 = fig.colorbar(scatter_mid2, cax=cbar_ax2)

# Set label on one colorbar only.
cbar_low.set_label(r"$\mu_{\rm tot}$ [mas yr$^{-1}$]")

# Hide ticks and labels on the second colorbar.
cbar_mid2.set_ticks([])
cbar_mid2.ax.tick_params(size=0)

# Plot trajectories.
for i in range(len(RA_SKA_mid2)):
    if pm_mag_SKA_mid2[i] > 50:
        ax.plot(
            RA_SKA_mid2_traj[i],
            DEC_SKA_mid2_traj[i],
            linestyle="-",
            linewidth=1.5,
            color="black",
            alpha=1,
            rasterized=True,
        )
for i in range(len(RA_SKA_low)):
    if pm_mag_SKA_low[i] > 50:
        ax.plot(
            RA_SKA_low_traj[i],
            DEC_SKA_low_traj[i],
            linestyle="-",
            linewidth=1.5,
            color="black",
            alpha=1,
            rasterized=True,
        )

# Add Galactic centre.
ax.plot(
    RA_galcen,
    DEC_galcen,
    marker="*",
    markeredgewidth=1,
    color="#800020",
    markersize=25,
    zorder=1,
)

ax.set_xlim(0.0, 360.0)
ax.set_ylim(-90.0, 36.0)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("DEC [deg]")

plt.savefig("pm_SKA_AA4.jpg", dpi=300, bbox_inches="tight")
plt.show()

## Determining new detections

Constrain observed sources to regions for Mid Band 2 and Low according to Survey Option 3 in Keane et al. (2025).

In [ ]:
mask_SKA_low = (df_atnf["DECJD"] < 36.0) & (np.abs(df_atnf["Gb"]) > 5.0)
mask_SKA_mid2 = (df_atnf["DECJD"] < 30.0) & (np.abs(df_atnf["Gb"]) < 5.0)

In [ ]:
# Extract longitude and latitude for observed pulsars in SKA regions.
l_obs_overlap_low = df_atnf[mask_SKA_low]["Gl"].to_numpy().astype(np.float64)
b_obs_overlap_low = df_atnf[mask_SKA_low]["Gb"].to_numpy().astype(np.float64)

l_obs_overlap_mid2 = df_atnf[mask_SKA_mid2]["Gl"].to_numpy().astype(np.float64)
b_obs_overlap_mid2 = df_atnf[mask_SKA_mid2]["Gb"].to_numpy().astype(np.float64)

l_obs_overlap_low[
    (l_obs_overlap_low > 180.0) & (l_obs_overlap_low < 360.0)
] = (
    l_obs_overlap_low[
        (l_obs_overlap_low > 180.0) & (l_obs_overlap_low < 360.0)
    ]
    - 360.0
)

l_obs_overlap_mid2[
    (l_obs_overlap_mid2 > 180.0) & (l_obs_overlap_mid2 < 360.0)
] = (
    l_obs_overlap_mid2[
        (l_obs_overlap_mid2 > 180.0) & (l_obs_overlap_mid2 < 360.0)
    ]
    - 360.0
)

In [ ]:
print("Observed sources in SKA-Low Sky coverage:", len(l_obs_overlap_low))
print(
    "Observed sources in SKA-Mid Band 2 Sky coverage:", len(l_obs_overlap_mid2)
)
print(
    "Additional detection with SKA-Low:",
    len(df_SKA_low) - len(l_obs_overlap_low),
)
print(
    "Additional detection with SKA-Mid Band 2:",
    len(df_SKA_mid2) - len(l_obs_overlap_mid2),
)
print(
    "Increase for SKA-Low:",
    np.round(len(df_SKA_low) / len(l_obs_overlap_low), 2),
)
print(
    "Increase for SKA-Mid Band 2:",
    np.round(len(df_SKA_mid2) / len(l_obs_overlap_mid2), 2),
)

## Analysing stars' brightness

### Mid region

In [ ]:
df_full_mid2_region = df_full[
    (df_full["DEC"]["[deg]"] < 30.0) & (np.abs(df_full["b"]["[deg]"]) < 5.0)
]

Extracting luminosity of all stars simulated in Mid region.

In [ ]:
L_radio_mid2_region = (
    df_full_mid2_region["L_radio_bol"]["[erg s^-1]"]
    .to_numpy()
    .astype(np.float64)
)

print(len(L_radio_mid2_region))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(L_radio_mid2_region),
    histtype="step",
    color="purple",
    lw=4,
    alpha=0.8,
    label=r"SKA-Mid region full simulated",
)

ax.set_yscale("log")

plt.xlabel(r"log$_{10}$ radio luminosity [erg s$^{-1}$]")
plt.ylabel(r"Number of stars")
plt.legend(frameon=True, loc=1)

plt.show()

Restricting flux analysis to those that are pointing at us.

In [ ]:
df_full_mid2_region_intercept = df_full_mid2_region[
    df_full_mid2_region["S_radio_bol"]["[erg s^-1 cm^(-2)]"] != 0.0
]

S_radio_bol_mid2_region = (
    df_full_mid2_region_intercept["S_radio_bol"]["[erg s^-1 cm^(-2)]"]
    .to_numpy()
    .astype(np.float64)
)

print(len(S_radio_bol_mid2_region))

print(
    np.round(len(S_radio_bol_mid2_region) / len(L_radio_mid2_region) * 100, 1)
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(S_radio_bol_mid2_region),
    histtype="step",
    color="purple",
    lw=4,
    alpha=0.8,
    label=r"SKA-Mid region intercepted",
)

ax.set_yscale("log")

plt.xlabel(r"log$_{10}$ bolometric radio flux [erg s$^{-1}$ cm$^{-2}$]")
plt.ylabel(r"Number of stars")
plt.legend(frameon=True, loc=1)

plt.show()

Mean period-averaged fluxes for the simulated SKA-Mid population.

In [ ]:
S_radio_mean_SKA_mid2 = (
    df_SKA_mid2["S_radio_obs_mean"]["[Jy]"].to_numpy().astype(np.float64)
)

print(len(S_radio_mean_SKA_mid2))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(S_radio_mean_SKA_mid2),
    histtype="step",
    color=colours[1],
    lw=4,
    alpha=0.8,
    label=r"SKA-Mid Band 2 observed",
)

ax.set_yscale("log")

plt.xlabel(r"log$_{10}$ radio flux [Jy]")
plt.ylabel(r"PDF of detected stars")
plt.legend(frameon=True, loc=1)

plt.show()

Convert bolometric fluxes for those sources that are observed with SKA-Mid to radio fluxes as a check.

In [ ]:
# Extract effective pulse width and periods of SKA Mid sources.
w_eff_SKA_mid2 = df_SKA_mid2["w_eff"]["[s]"].to_numpy().astype(np.float64)
P_SKA_mid2 = df_SKA_mid2["P"]["[s]"].to_numpy().astype(np.float64)

In [ ]:
# Extract bolometric fluxes and intrinsic pulse widths of SKA Mid sources.
SKA_mid2_indices = np.stack(
    df_SKA_mid2["NS_idx"].to_numpy().astype(np.int64), axis=1
)[0]

SKA_mid2_pulsars_S_radio_bol = (
    df_full.iloc[SKA_mid2_indices]["S_radio_bol"]["[erg s^-1 cm^(-2)]"]
    .to_numpy()
    .astype(np.float64)
)

print(SKA_mid2_pulsars_S_radio_bol)

w_int_SKA_mid2 = (
    df_full.iloc[SKA_mid2_indices]["w_int"]["[s]"]
    .to_numpy()
    .astype(np.float64)
)

print(w_int_SKA_mid2)

In [ ]:
import pypopsyn.simulator.multiband_emission.emission_radio as er

In [ ]:
# Convert bolometric flux to flux density at SKA-Mid central frequency.
spectral_index = (
    df_full.iloc[SKA_mid2_indices]["spectral_index"][" "]
    .to_numpy()
    .astype(np.float64)
)
f_central_mid = 1.4e9

S_radio_f_mid2 = er.flux_density_radio(
    SKA_mid2_pulsars_S_radio_bol, spectral_index, f_central_mid
)

In [ ]:
# Account for different pulse width while keeping fluence constant to determine observed flux.
S_radio_f_obs_mid2 = S_radio_f_mid2 * w_int_SKA_mid2 / w_eff_SKA_mid2

In [ ]:
# Period average to obtain mean observed flux.
S_radio_f_obs_mean_mid2 = S_radio_f_obs_mid2 * w_eff_SKA_mid2 / P_SKA_mid2

In [ ]:
print(max(S_radio_f_obs_mean_mid2 / S_radio_mean_SKA_mid2))
print(min(S_radio_f_obs_mean_mid2 / S_radio_mean_SKA_mid2))

Highlight fraction of observed sources with SKA-Mid for full population pointing at us.

In [ ]:
S_radio_bol_mid2_region

In [ ]:
bins_S_radio_bol = np.linspace(-28, -10, 19)
print(bins_S_radio_bol)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(S_radio_bol_mid2_region),
    bins=bins_S_radio_bol,
    histtype="step",
    color="purple",
    lw=4,
    alpha=0.8,
    label=r"SKA-Mid Band 2 intercepted",
)

ax.hist(
    np.log10(SKA_mid2_pulsars_S_radio_bol),
    bins=bins_S_radio_bol,
    histtype="step",
    color=colours[1],
    lw=4,
    alpha=0.8,
    label=r"SKA-Mid Band 2 observed",
)

ax.set_yscale("log")

plt.xlabel(r"log$_{10}$ bolometric radio flux [erg s$^{-1}$ cm$^{-2}$]")
plt.ylabel(r"Number of stars")
plt.legend(frameon=True, loc=1)

plt.show()

### Low region

In [ ]:
df_full_low_region = df_full[
    (df_full["DEC"]["[deg]"] < 36.0) & (np.abs(df_full["b"]["[deg]"]) > 5.0)
]

In [ ]:
df_full_low_region_intercept = df_full_low_region[
    df_full_low_region["S_radio_bol"]["[erg s^-1 cm^(-2)]"] != 0.0
]

S_radio_bol_low_region = (
    df_full_low_region_intercept["S_radio_bol"]["[erg s^-1 cm^(-2)]"]
    .to_numpy()
    .astype(np.float64)
)

print(len(S_radio_bol_low_region))

In [ ]:
S_radio_mean_SKA_low = (
    df_SKA_low["S_radio_obs_mean"]["[Jy]"].to_numpy().astype(np.float64)
)

print(len(S_radio_mean_SKA_low))

print(max(S_radio_mean_SKA_low))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
ax.set_title(f"AA4 configuration", fontsize=MEDIUM_SIZE)

ax.hist(
    np.log10(S_radio_mean_SKA_low),
    histtype="step",
    color=colours[0],
    lw=4,
    alpha=0.8,
    label=r"SKA-Low",
)

ax.hist(
    np.log10(S_radio_mean_SKA_mid2),
    histtype="step",
    color=colours[1],
    lw=4,
    alpha=0.8,
    label=r"SKA-Mid Band 2",
)

ax.set_yscale("log")

plt.xlabel(r"log$_{10}$ radio flux [Jy]")
plt.ylabel(r"PDF of detected stars")
plt.legend(frameon=True, loc=1)

plt.show()

In [ ]:
# Extract effective pulse width and periods of SKA Low sources.
w_eff_SKA_low = df_SKA_low["w_eff"]["[s]"].to_numpy().astype(np.float64)
P_SKA_low = df_SKA_low["P"]["[s]"].to_numpy().astype(np.float64)

# Extract bolometric fluxes and intrinsic pulse widths of SKA Low sources.
SKA_low_indices = np.stack(
    df_SKA_low["NS_idx"].to_numpy().astype(np.int64), axis=1
)[0]

SKA_low_pulsars_S_radio_bol = (
    df_full.iloc[SKA_low_indices]["S_radio_bol"]["[erg s^-1 cm^(-2)]"]
    .to_numpy()
    .astype(np.float64)
)

print(SKA_low_pulsars_S_radio_bol)

w_int_SKA_low = (
    df_full.iloc[SKA_low_indices]["w_int"]["[s]"].to_numpy().astype(np.float64)
)

print(w_int_SKA_low)

In [ ]:
import pypopsyn.simulator.multiband_emission.emission_radio as er

# Convert bolometric flux to flux density at SKA Mid central frequency.
spectral_index = (
    df_full.iloc[SKA_low_indices]["spectral_index"][" "]
    .to_numpy()
    .astype(np.float64)
)
f_central_low = 190.0e6

S_radio_f_low = er.flux_density_radio(
    SKA_low_pulsars_S_radio_bol, spectral_index, f_central_low
)

# Account for different pulse width while keeping fluence constant to determine observed flux.
S_radio_f_obs_low = S_radio_f_low * w_int_SKA_low / w_eff_SKA_low

# Period average to obtain mean observed flux.
S_radio_f_obs_mean_low = S_radio_f_obs_low * w_eff_SKA_low / P_SKA_low

In [ ]:
print(max(S_radio_f_obs_mean_low / S_radio_mean_SKA_low))
print(min(S_radio_f_obs_mean_low / S_radio_mean_SKA_low))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
ax.set_title(f"AA4 configuration", fontsize=MEDIUM_SIZE)

ax.hist(
    np.log10(S_radio_bol_low_region),
    bins=bins_S_radio_bol,
    histtype="step",
    color="purple",
    lw=4,
    alpha=0.8,
    label=r"SKA-Low region intercepted",
)

ax.hist(
    np.log10(SKA_low_pulsars_S_radio_bol),
    bins=bins_S_radio_bol,
    histtype="step",
    color=colours[0],
    lw=4,
    alpha=0.8,
    label=r"SKA-Low observed",
)

ax.set_yscale("log")

plt.xlabel(r"log$_{10}$ bolometric radio flux [erg s$^{-1}$ cm$^{-2}$]")
plt.ylabel(r"Number of stars")
plt.legend(frameon=True, loc=1)

plt.show()

Plotting flux vs period derivative.

In [ ]:
Pdot_SKA_mid2 = df_SKA_mid2["P_dot"]["[s s^-1]"].to_numpy().astype(np.float64)
Pdot_SKA_low = df_SKA_low["P_dot"]["[s s^-1]"].to_numpy().astype(np.float64)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 14))
ax.set_title(f"AA4 configuration", fontsize=MEDIUM_SIZE)

ax.scatter(
    Pdot_SKA_low,
    S_radio_f_obs_mean_low,
    marker="o",
    color=colours[0],
    s=40,
    alpha=1,
    label=r"SKA-Low",
)

ax.scatter(
    Pdot_SKA_mid2,
    S_radio_f_obs_mean_mid2,
    marker="o",
    color=colours[1],
    s=40,
    alpha=1,
    label=r"SKA-Mid Band 2",
)

ax.set_xscale("log")
ax.set_yscale("log")

plt.xlabel(r"Period derivative [s/s]")
plt.ylabel(r"Radio flux [Jy]")
plt.legend(frameon=True, loc=2)

plt.tight_layout()
plt.show()

## Counting young pulsars

Combine two SKA arrays. As noted above due to the survey configuration in Option 3, these are unique.

In [ ]:
df_SKA_combined = pd.concat([df_SKA_mid2, df_SKA_low])

print(len(df_SKA_combined))

In [ ]:
df_SKA_combined_tau_c = (
    df_SKA_combined["P"]["[s]"]
    / (2 * df_SKA_combined["P_dot"]["[s s^-1]"])
    / 3600
    / 24
    / 365
)

In [ ]:
df_SKA_combined_tau_c

In [ ]:
sum(df_SKA_combined_tau_c.values < 1e6)